In [194]:
from pyspark.sql import SparkSession
from pyspark.sql import types
import pyspark.sql.functions as F

In [195]:
# Create a spark session
spark = SparkSession.builder.master("local[*]").appName("taxi-rides-app").getOrCreate()

print("[INFO] Starting spark session")

if not spark.version:
    print("[ERROR] Could not start SPARK session. Exiting program.")
    import os

    exit(1)

[INFO] Starting spark session


In [196]:
from common.config import get_root_path

# Declare dataset paths
DATA_PATH = get_root_path() / "data"
TAXI_PATH = DATA_PATH / "taxi"
DATASET_CLEAN_PATH = TAXI_PATH / "clean" / "yellow" / "2025" / "11"
DATASET_REPORT_PATH = TAXI_PATH / "report"

In [197]:
print(f"[INFO] Reading clean dataset yellow/2025/11")
dataset_file_path = str(DATASET_CLEAN_PATH)
df = spark.read.parquet(dataset_file_path).repartition(4)

print(f"[INFO] Reading taxi zone lookup dataset")
lookup_file_path = str(DATA_PATH / "taxi_zone_lookup.csv")
lookup_df = spark.read.csv(lookup_file_path, header=True)

[INFO] Reading clean dataset yellow/2025/11
[INFO] Reading taxi zone lookup dataset


In [198]:
lookup_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [199]:
join_col = df["pickup_location_id"] == lookup_df["LocationID"]
df_joined = df.join(lookup_df, on=join_col)
df_joined.show()

+---------+-------------------+-------------------+------------------+------------+------------------+-------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+------------+-----------+----------+---------+--------------------+------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|rate_code_id|pickup_location_id|dropoff_location_id|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|service_type|airport_fee|LocationID|  Borough|                Zone|service_zone|
+---------+-------------------+-------------------+------------------+------------+------------------+-------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+------

In [200]:
df2 = df \
    .withColumn("pickup_ts", F.to_timestamp("pickup_datetime")) \
    .withColumn("dropoff_ts", F.to_timestamp("dropoff_datetime")) \
    .withColumn(
        "trip_hours",
        (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 3600.0
    )

# longest trip
df2.orderBy(F.col("trip_hours").desc()).limit(1).select("dropoff_datetime", "pickup_datetime", "trip_hours").show()

+-------------------+-------------------+-----------------+
|   dropoff_datetime|    pickup_datetime|       trip_hours|
+-------------------+-------------------+-----------------+
|2025-11-30 15:01:00|2025-11-26 20:22:12|90.64666666666666|
+-------------------+-------------------+-----------------+



In [201]:
df3 = df_joined.groupBy("Zone").count().orderBy(F.col("count").asc())

df3.show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Port Richmond|    3|
|   Rossville/Woodrow|    4|
| Green-Wood Cemetery|    4|
|         Great Kills|    4|
|       Rikers Island|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|New Dorp/Midland ...|   14|
|       West Brighton|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows


In [202]:
print(f"[INFO] Processing report: homework")

nov_15_rides_count = df.filter(F.extract(F.lit("D"), df.pickup_datetime) == F.lit("15"))

report_df = spark.createDataFrame(
    [
        ("spark-version", spark.version),
        ("parquet-avg-size", 25),
        ("nov_15_rides_count", nov_15_rides_count.count()),
        ("longest_trip_in_hours", 90.6),
        ("spark-ui-port", 4040),
        ("least-frequent-pickup-zone", "Arden Heights")
    ],
    ["name", "value"],
)

[INFO] Processing report: homework


In [203]:
print(f"[INFO] Loading report: homework")
output_path = str(DATASET_REPORT_PATH / "homework")
report_df.repartition(1).write.parquet(
    path=output_path,
    mode="overwrite",
)

[INFO] Loading report: homework


In [205]:
report_df.show()

+--------------------+-------------+
|                name|        value|
+--------------------+-------------+
|       spark-version|        4.1.1|
|    parquet-avg-size|           25|
|  nov_15_rides_count|       162604|
|longest_trip_in_h...|         90.6|
|       spark-ui-port|         4040|
|least-frequent-pi...|Arden Heights|
+--------------------+-------------+



In [204]:
# spark.stop()